# EM rollout flux comparison: GKW vs gyaradax

This notebook compares persisted gyaradax nonlinear EM rollouts against generated GKW diagnostics. It uses physical-time alignment: gyaradax diagnostic columns are linearly interpolated onto the GKW `time.dat` grid over the common time interval.

Data roots are external and are not committed to the repo:

- GKW: `/local00/bioinf/volkmann/gyrokinetics/em_validation`
- gyaradax rollouts: `/local00/bioinf/volkmann/gyrokinetics/em_validation/gyaradax_rollouts`


In [ ]:
from __future__ import annotations

from pathlib import Path
from io import StringIO
import re

import numpy as np
try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None
import matplotlib.pyplot as plt

GKW_ROOT = Path('/local00/bioinf/volkmann/gyrokinetics/em_validation')
ROLLOUT_ROOT = GKW_ROOT / 'gyaradax_rollouts'

CASES = [
    'observables__nonlinear_apar_b005',
    'observables__nonlinear_apar_b01',
    'observables__nonlinear_apar_bpar_b005',
    'observables__nonlinear_apar_bpar_b01',
    'observables__nonlinear_bpar_only_b005',
    'observables__nonlinear_bpar_only_b01',
]

DATASETS = {
    'fluxes': ('fluxes.npy', 'fluxes.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
    'fluxes_em': ('fluxes_em.npy', 'fluxes_em.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
    'fluxes_bpar': ('fluxes_bpar.npy', 'fluxes_bpar.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
}

def load_gkw_table(path: Path) -> np.ndarray | None:
    if not path.exists():
        return None
    raw_lines = [line for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not raw_lines:
        return None
    text = '\n'.join(raw_lines)
    # GKW es13.5 can overflow 3-digit exponents as e.g. 1.14010+100.
    text = re.sub(r'(?<=[0-9.])([+-][0-9]{2,})(?=\s|$)', r'E\1', text)
    arr = np.asarray(np.loadtxt(StringIO(text)))
    if arr.ndim == 0:
        return arr.reshape(1, 1)
    if arr.ndim == 1:
        return arr.reshape(1, -1) if len(raw_lines) == 1 else arr.reshape(-1, 1)
    return arr

def rollout_key_to_gkw_dir(case_key: str) -> Path:
    if '__' in case_key:
        return GKW_ROOT.joinpath(*case_key.split('__'))
    return GKW_ROOT / 'observables' / case_key

def load_case(case_key: str):
    rollout_dir = ROLLOUT_ROOT / case_key
    gkw_dir = rollout_key_to_gkw_dir(case_key)
    if not rollout_dir.exists():
        raise FileNotFoundError(f'Missing rollout dir: {rollout_dir}')
    if not gkw_dir.exists():
        raise FileNotFoundError(f'Missing GKW dir: {gkw_dir}')
    gy_time = np.load(rollout_dir / 'time.npy').reshape(-1)
    gkw_time = load_gkw_table(gkw_dir / 'time.dat')[:, 0]
    return rollout_dir, gkw_dir, gy_time, gkw_time

def common_time_grid(gy_time: np.ndarray, gkw_time: np.ndarray) -> np.ndarray:
    start = max(float(np.nanmin(gy_time)), float(np.nanmin(gkw_time)))
    end = min(float(np.nanmax(gy_time)), float(np.nanmax(gkw_time)))
    mask = (gkw_time >= start) & (gkw_time <= end)
    return gkw_time[mask]

def interp_columns(src_t: np.ndarray, values: np.ndarray, dst_t: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim == 1:
        values = values.reshape(-1, 1)
    cols = []
    for i in range(values.shape[1]):
        cols.append(np.interp(dst_t, src_t, values[:, i]))
    return np.stack(cols, axis=1)

def aligned_dataset(case_key: str, dataset: str):
    rollout_dir, gkw_dir, gy_time, gkw_time = load_case(case_key)
    npy_name, dat_name, labels = DATASETS[dataset]
    gy_path = rollout_dir / npy_name
    gkw_path = gkw_dir / dat_name
    if not gy_path.exists() or not gkw_path.exists():
        return None
    gy = np.load(gy_path)
    gkw = load_gkw_table(gkw_path)
    t = common_time_grid(gy_time, gkw_time)
    gkw_mask = np.isin(gkw_time, t)
    gy_i = interp_columns(gy_time, gy, t)
    gkw_i = gkw[gkw_mask, :gy_i.shape[1]]
    ncols = min(gy_i.shape[1], gkw_i.shape[1])
    return t, gy_i[:, :ncols], gkw_i[:, :ncols], labels[:ncols]

def metrics(gy: np.ndarray, gkw: np.ndarray, floor: float = 1e-30) -> dict:
    denom = np.maximum(np.abs(gkw), floor)
    rel = np.abs(gy - gkw) / denom
    log_corrs = []
    for col in range(gy.shape[1]):
        mask = (np.abs(gy[:, col]) > floor) & (np.abs(gkw[:, col]) > floor)
        if mask.sum() >= 3:
            x = np.log(np.abs(gy[mask, col]))
            y = np.log(np.abs(gkw[mask, col]))
            if np.std(x) > 0 and np.std(y) > 0:
                log_corrs.append(np.corrcoef(x, y)[0, 1])
    sign_mask = (np.abs(gy) > floor) & (np.abs(gkw) > floor)
    return {
        'max_gkw': float(np.nanmax(np.abs(gkw))),
        'max_gyaradax': float(np.nanmax(np.abs(gy))),
        'median_rel': float(np.nanmedian(rel)),
        'max_rel': float(np.nanmax(rel)),
        'log_corr_median': float(np.nanmedian(log_corrs)) if log_corrs else np.nan,
        'sign_agreement': float(np.mean(np.sign(gy[sign_mask]) == np.sign(gkw[sign_mask]))) if sign_mask.any() else np.nan,
    }


## Summary table

The table below uses time-aligned diagnostics over the common physical-time interval. `median_rel` is the median elementwise relative difference across all compared rows and columns. `log_corr` is the median columnwise correlation of log-amplitudes.


In [ ]:
rows = []
for case in CASES:
    for dataset in DATASETS:
        aligned = aligned_dataset(case, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        m = metrics(gy, gkw)
        rows.append({
            'case': case.replace('observables__', ''),
            'dataset': dataset,
            'points': len(t),
            't_start': t[0],
            't_end': t[-1],
            **m,
        })
if pd is not None:
    summary = pd.DataFrame(rows)
    display(summary)
else:
    summary = rows
    for row in rows:
        print(row)


## Plot helper


In [ ]:
def plot_case(case_key: str, datasets=('fluxes', 'fluxes_em', 'fluxes_bpar')):
    display_name = case_key.replace('observables__', '')
    for dataset in datasets:
        aligned = aligned_dataset(case_key, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        ncols = gy.shape[1]
        fig, axes = plt.subplots(ncols, 1, figsize=(12, 2.2 * ncols), sharex=True)
        axes = np.atleast_1d(axes)
        for i, ax in enumerate(axes):
            ax.plot(t, gkw[:, i], label='GKW', lw=1.6)
            ax.plot(t, gy[:, i], label='gyaradax', lw=1.2, ls='--')
            ax.set_ylabel(labels[i])
            ax.grid(True, alpha=0.25)
            if i == 0:
                ax.set_title(f'{display_name}: {dataset}')
                ax.legend(loc='best')
        axes[-1].set_xlabel('physical time')
        plt.tight_layout()
        plt.show()

def plot_case_abs(case_key: str, datasets=('fluxes', 'fluxes_em', 'fluxes_bpar')):
    display_name = case_key.replace('observables__', '')
    for dataset in datasets:
        aligned = aligned_dataset(case_key, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(t, np.max(np.abs(gkw), axis=1), label='GKW max |columns|', lw=1.8)
        ax.plot(t, np.max(np.abs(gy), axis=1), label='gyaradax max |columns|', lw=1.4, ls='--')
        ax.set_yscale('log')
        ax.set_title(f'{display_name}: {dataset} max absolute flux')
        ax.set_xlabel('physical time')
        ax.set_ylabel('max |flux column|')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()


## Case: nonlinear_apar_b005


In [ ]:
plot_case('observables__nonlinear_apar_b005')
plot_case_abs('observables__nonlinear_apar_b005')


## Case: nonlinear_apar_b01


In [ ]:
plot_case('observables__nonlinear_apar_b01')
plot_case_abs('observables__nonlinear_apar_b01')


## Case: nonlinear_apar_bpar_b005


In [ ]:
plot_case('observables__nonlinear_apar_bpar_b005')
plot_case_abs('observables__nonlinear_apar_bpar_b005')


## Case: nonlinear_apar_bpar_b01


In [ ]:
plot_case('observables__nonlinear_apar_bpar_b01')
plot_case_abs('observables__nonlinear_apar_bpar_b01')


## Case: nonlinear_bpar_only_b005


In [ ]:
plot_case('observables__nonlinear_bpar_only_b005')
plot_case_abs('observables__nonlinear_bpar_only_b005')


## Case: nonlinear_bpar_only_b01


In [ ]:
plot_case('observables__nonlinear_bpar_only_b01')
plot_case_abs('observables__nonlinear_bpar_only_b01')
